# Interface-state charge and Fermi-level pinning — release 0.5

Release 0.5 solves the surface charge balance

$$Q_\mathrm{ext}(\psi_s)+Q_{it}(\psi_s,\Delta n)+Q_{sc}(\psi_s,\Delta n)=0.$$

The same distributed defects now contribute both SRH recombination and charge.
States below an adjustable charge-neutrality level (CNL) are donor-like
(positive when empty); states above it are acceptor-like (negative when
occupied). Their steady-state occupancy is calculated from electron and hole
capture and emission rates.

The CNL and donor/acceptor partition are **sensitivity parameters**, not claimed
material constants. Degenerate carrier populations are used in Poisson's
equation, but fully degenerate trap kinetics with Pauli blocking is not yet
included.

In [ ]:
import sys
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

root = Path.cwd()
if not (root / "src").exists(): root = root.parent
sys.path.insert(0, str(root / "src"))
from surpass import (BulkModel, InterfaceChargeModel, InterfaceDefectModel,
                        SurfaceSRHModel, Wafer)

try:
    import scienceplots
    plt.style.use(["science", "notebook", "no-latex"])
except ImportError:
    plt.style.use("default")
plt.rcParams.update({"figure.dpi": 120, "font.size": 10})

## 1. Editable interface assumptions

In [ ]:
qf_values = np.linspace(-6e12, 6e12, 61)
dit_values = [1e9, 1e11, 1e13, 1e15]
cnl_fraction = 0.50
transition_width_ev = 0.02
delta_probe_cm3 = 1e11

wafers = {
    "n-InP": Wafer("InP", donor_cm3=8.5e18, thickness_um=620),
    "p-InP": Wafer("InP", acceptor_cm3=5.1e18, thickness_um=620),
}
charge_model = InterfaceChargeModel(cnl_fraction, transition_width_ev)

def make_model(wafer, qf, dit, charge=True):
    defects = InterfaceDefectModel(
        dit_mid_ev1_cm2=dit, sigma_n_cm2=1e-14, sigma_p_cm2=1e-13,
        shape="constant", energy_points=301)
    return SurfaceSRHModel(BulkModel(wafer), qf, defects,
                           interface_charge=charge_model if charge else None)

## 2. Pinning and charge partition

The plotted trap-control fraction is
$C_{it}/(C_{it}+C_{sc})$, using local differential capacitances. It is a useful
electrostatic diagnostic, but it is not the classical Schottky-barrier slope
parameter.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(10, 7), sharex=True)
rows = []
for column, (label, wafer) in enumerate(wafers.items()):
    for dit in dit_values:
        states = [make_model(wafer, qf, dit).state(0.0) for qf in qf_values]
        axes[0, column].plot(qf_values/1e12, [s.psi_surface_v for s in states],
                             label=f"$D_{{it}}={dit:.0e}$")
        axes[1, column].plot(qf_values/1e12,
                             [s.trap_control_fraction for s in states])
        for qf, state in zip(qf_values, states):
            rows.append({"wafer": label, "Dit": dit, "Qext/q": qf,
                         "psi_s": state.psi_surface_v,
                         "Qit/q": state.q_it_number_cm2,
                         "Qsc/q": state.q_sc_number_cm2,
                         "balance/q": state.charge_balance_number_cm2,
                         "trap control": state.trap_control_fraction})
    axes[0, column].set(title=label, ylabel="$\\psi_s$ (V)")
    axes[1, column].set(xlabel="$Q_{ext}/q$ ($10^{12}$ cm$^{-2}$)",
                         ylabel="$C_{it}/(C_{it}+C_{sc})$")
axes[0, 0].legend(fontsize=8)
fig.tight_layout()
charge_df = pd.DataFrame(rows)
print("Maximum absolute charge-balance error:",
      f"{charge_df['balance/q'].abs().max():.3g} elementary charges cm^-2")

## 3. The same traps affect recombination and electrostatics

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.7))
for ax, (label, wafer) in zip(axes, wafers.items()):
    for enabled, style in [(False, "--"), (True, "-")]:
        states = [make_model(wafer, qf, 1e13, charge=enabled).state(delta_probe_cm3)
                  for qf in qf_values]
        ax.semilogy(qf_values/1e12, [s.recombination_cm2_s for s in states],
                    ls=style, label="$Q_{it}$ on" if enabled else "$Q_{it}=0$")
    ax.set(title=label, xlabel="$Q_{ext}/q$ ($10^{12}$ cm$^{-2}$)",
           ylabel="$U_s$ (cm$^{-2}$ s$^{-1}$)")
    ax.legend()
fig.tight_layout()

## 4. CNL sensitivity

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.7))
for ax, (label, wafer) in zip(axes, wafers.items()):
    for cnl in (0.35, 0.50, 0.65):
        local_charge = InterfaceChargeModel(cnl, transition_width_ev)
        defects = InterfaceDefectModel(dit_mid_ev1_cm2=1e13, energy_points=301)
        states = [SurfaceSRHModel(BulkModel(wafer), qf, defects,
                                  interface_charge=local_charge).state(0.0)
                  for qf in qf_values]
        ax.plot(qf_values/1e12, [s.psi_surface_v for s in states],
                label=f"CNL = {cnl:.2f}$E_g$")
    ax.set(title=label, xlabel="$Q_{ext}/q$ ($10^{12}$ cm$^{-2}$)",
           ylabel="$\\psi_s$ (V)")
    ax.legend(fontsize=8)
fig.tight_layout()

## 5. Architecture for a later biased gate

`SurfaceSRHModel` accepts any external boundary object that implements
`charge_c_cm2(psi_surface_v)`. The present fixed-charge boundary returns a
constant. A later metal/oxide gate can instead return, for example,

$$Q_g=C_{ox}(V_g-V_{fb}-\psi_s),$$

and the same root solve will include gate bias, work-function difference,
oxide thickness, and fixed oxide charge. That later implementation should add
finite dielectric stacks and a clear voltage/reference convention; it is not
silently approximated in release 0.5.

## Interpretation

- With small $D_{it}$, semiconductor space charge balances most external
  charge and $\psi_s$ responds strongly.
- With large $D_{it}$, trap charge absorbs more of the perturbation and the
  surface Fermi level approaches the chosen CNL: this is pinning.
- Because the same $D_{it}(E)$ and capture coefficients set both occupancy and
  recombination, changing them can shift both band bending and surface loss.
- Quantitative extraction therefore needs independent constraints on the CNL,
  donor/acceptor character, capture cross sections, and fixed charge.